# Imports

In [142]:
import torch 
import torch.nn as nn
import pandas as pd

from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
print(torch.__version__)

2.7.0.dev20250304+cu128


In [143]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load data

In [144]:
df = pd.read_csv('../data/prepared_data.csv', index_col='id')
df

,Gender,Age,Academic_Pressure,Work_Pressure,CGPA,Study_Satisfaction,Job_Satisfaction,had_suicidal_thoughts,Work_Study_Hours,Financial_Stress,...,Degree_M.Tech,Degree_MA,Degree_MBA,Degree_MBBS,Degree_MCA,Degree_MD,Degree_ME,Degree_MHM,Degree_MSc,Degree_PhD
id,,,,,,,,,,,,,,,,,,,,,
2,0,1.463374,1.345217,-0.00978,0.893358,-0.693493,-0.015345,1,-1.121284,-1.489038,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,1,-0.371245,-0.826029,-0.00978,-1.194180,1.510716,-0.015345,0,-1.121284,-0.793282,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
26,0,1.055681,-0.102280,-0.00978,-0.425803,1.510716,-0.015345,0,0.497192,-1.489038,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
30,1,0.444141,-0.102280,-0.00978,-1.404974,-0.693493,-0.015345,1,-0.851538,1.293986,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
32,1,-0.167398,0.621468,-0.00978,0.322175,0.041243,-0.015345,1,-1.660776,-1.489038,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
140685,1,0.240295,1.345217,-0.00978,-1.296177,1.510716,-0.015345,1,-0.042300,-1.489038,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
140686,0,0.240295,-0.826029,-0.00978,1.185749,0.041243,-0.015345,0,-1.930522,-0.097526,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
140689,0,1.055681,-0.102280,-0.00978,-0.711394,0.775980,-0.015345,0,1.306430,-0.793282,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


# Separate data

In [145]:
X = df.drop(['Depression'], axis = 1)
y = df['Depression']

In [146]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.35, random_state=42)
train_X = torch.from_numpy(X_train.values).to(torch.float32)
train_y = torch.from_numpy(y_train.values).type_as(train_X)
test_X = torch.from_numpy(X_test.values).to(torch.float32)
test_y = torch.from_numpy(y_test.values).type_as(test_X)

In [147]:
BATCH_SIZE = 256

In [148]:
train_DS = TensorDataset(train_X, train_y)
test_DS = TensorDataset(test_X, test_y)

In [149]:
train_DL = DataLoader(train_DS, batch_size=BATCH_SIZE, shuffle=True)
test_DL = DataLoader(test_DS, batch_size=BATCH_SIZE, shuffle=True)

In [150]:
class DepressionModel(nn.Module):
    def __init__(self, in_features:int):
        super().__init__()
        self.ln1 = nn.Linear(in_features, in_features*4)
        self.ln2 = nn.Linear(in_features*4, in_features)
        self.ln3 = nn.Linear(in_features, in_features//2)
        self.ln4 = nn.Linear(in_features//2, in_features//4)
        self.ln5 = nn.Linear(in_features//4, 1)
        self.relu = nn.ReLU()
    def forward(self, x):
        x = self.ln1(x)
        x = self.relu(x)
        x = self.ln2(x)
        x = self.relu(x)
        x = self.ln3(x)
        x = self.relu(x)
        x = self.ln4(x)
        x = self.relu(x)
        x = self.ln5(x)
        return x

In [151]:
model = DepressionModel(45).to(DEVICE)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = Adam(params=model.parameters(), lr=0.00001)


epochs = 300
for epoch in range(epochs):
    train_loss = 0
    train_acc = 0
    for batch, (X, y) in enumerate(train_DL):
        model.train()
        X, y = X.to(DEVICE), y.to(DEVICE) 
        pred = model(X)
        y = y.unsqueeze(1)
        loss = loss_fn(pred, y)
        train_loss += loss
        predictions = (pred > 0.5).float()
        train_acc += (predictions == y).sum().item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss /= len(train_DL)
    train_acc /= len(train_DL.dataset)

    test_loss_total = 0
    test_acc = 0
    
    model.eval()

    with torch.inference_mode():
        for batch, (X_test, y_test) in enumerate(test_DL):
          # Make sure test data on CPU
            X_test, y_test = X_test.to(DEVICE), y_test.to(DEVICE)
            test_pred = model(X_test)
            y_test = y_test.unsqueeze(1)
            test_loss = loss_fn(test_pred, y_test)
            test_loss_total += test_loss
            predictions = (test_pred > 0.5).float()
            test_acc += (predictions == y_test).sum().item()

        test_loss_total /= len(test_DL)
        test_acc /= len(test_DL.dataset)

    if epoch % 20 == 0:
      print(
          f"Epoch: {epoch} | Loss: {train_loss:.3f} | Train Acc: {train_acc:.3f} | Test loss: {test_loss_total:.3f} | Test Acc: {test_acc:.3f}"
      )

Epoch: 0 | Loss: 0.705 | Train Acc: 0.413 | Test loss: 0.703 | Test Acc: 0.418
Epoch: 20 | Loss: 0.648 | Train Acc: 0.413 | Test loss: 0.645 | Test Acc: 0.418
Epoch: 40 | Loss: 0.482 | Train Acc: 0.800 | Test loss: 0.476 | Test Acc: 0.803
Epoch: 60 | Loss: 0.385 | Train Acc: 0.827 | Test loss: 0.380 | Test Acc: 0.829
Epoch: 80 | Loss: 0.360 | Train Acc: 0.838 | Test loss: 0.356 | Test Acc: 0.839
Epoch: 100 | Loss: 0.351 | Train Acc: 0.843 | Test loss: 0.352 | Test Acc: 0.839
Epoch: 120 | Loss: 0.348 | Train Acc: 0.843 | Test loss: 0.349 | Test Acc: 0.843
Epoch: 140 | Loss: 0.347 | Train Acc: 0.844 | Test loss: 0.349 | Test Acc: 0.844
Epoch: 160 | Loss: 0.347 | Train Acc: 0.843 | Test loss: 0.349 | Test Acc: 0.844
Epoch: 180 | Loss: 0.346 | Train Acc: 0.844 | Test loss: 0.346 | Test Acc: 0.844
Epoch: 200 | Loss: 0.346 | Train Acc: 0.844 | Test loss: 0.344 | Test Acc: 0.844
Epoch: 220 | Loss: 0.345 | Train Acc: 0.844 | Test loss: 0.346 | Test Acc: 0.845
Epoch: 240 | Loss: 0.345 | Train A